# Stage D ETF All-Weather Snapshot Manual Validation

Manual checks for `derived.etf_aw_rebalance_snapshot` parquet outputs. Run from the repository root.


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
SNAPSHOT_ROOT = PROJECT_ROOT / "data/lakehouse/derived/derived.etf_aw_rebalance_snapshot"

snapshot_files = sorted(SNAPSHOT_ROOT.glob("**/*.parquet"))
snapshot_files


In [ ]:
if not snapshot_files:
    raise FileNotFoundError(f"No Stage D snapshot parquet files under {SNAPSHOT_ROOT}")

con = duckdb.connect()
df = con.execute(
    f"""
    select *
    from read_parquet('{SNAPSHOT_ROOT.as_posix()}/**/*.parquet')
    order by rebalance_date, sleeve_code
    """
).df()

df.head()


In [ ]:
status_summary = df.groupby("data_status").size().rename("rows").reset_index()
status_summary


In [ ]:
rebalance_summary = df.groupby("rebalance_date").agg(
    rows=("sleeve_code", "count"),
    sleeves=("sleeve_code", "nunique"),
    complete=("data_status", lambda values: (values == "complete").sum()),
    partial=("data_status", lambda values: (values == "partial").sum()),
    missing=("data_status", lambda values: (values == "missing").sum()),
    stale=("data_status", lambda values: (values == "stale").sum()),
)
rebalance_summary


In [ ]:
df[[
    "calendar_month",
    "rebalance_date",
    "sleeve_code",
    "sleeve_role",
    "return_1m",
    "return_3m",
    "return_6m",
    "volatility_3m",
    "max_drawdown_6m",
    "data_status",
]]
